In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### DATA LOADING AND UNDERSTANDING

In [2]:
file_path = "..\Data\Raw\online_retail_II.xlsx"
sheets = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)

df.head()

<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
<>:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
C:\Users\Ade\AppData\Local\Temp\ipykernel_9408\712113259.py:1: SyntaxWarning: "\D" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\D"? A raw string is also an option.
  file_path = "..\Data\Raw\online_retail_II.xlsx"


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
# Jumlah baris dan kolom pada dataset
print("Database shape : ", df.shape)

# List kolom pada dataset
print("\nColumns:")
for c in df.columns:
    print("-", c)

# Tipe data dari tiap kolom di dataset
print("\nDtypes:")
display(df.dtypes.to_frame("dtype"))

# Check missing value di tiap kolom
print("\nMissing values per column:")
display(df.isnull().sum().to_frame("missing").sort_values("missing", ascending=False))
# Check duplikasi data di dataset
print("\nDuplicate rows:", int(df.duplicated().sum()))

Database shape :  (1067371, 8)

Columns:
- Invoice
- StockCode
- Description
- Quantity
- InvoiceDate
- Price
- Customer ID
- Country

Dtypes:


,dtype
Invoice,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
Price,float64
Customer ID,float64
Country,str



Missing values per column:


,missing
Customer ID,243007
Description,4382
StockCode,0
Invoice,0
Quantity,0
InvoiceDate,0
Price,0
Country,0



Duplicate rows: 34335


### DATA PREPROCESSING

In [4]:
missing_value = pd.DataFrame(
    {
        "missing_count": df.isnull().sum(),
        "missing_percentage": round(df.isnull().mean()*100,2)
    }
)
display(missing_value.sort_values("missing_percentage",ascending =False))

# i'm not dropping the missing value, because the missing value only in customer_id and description column. 
# so its still have a valuable transaction data in its row

,missing_count,missing_percentage
Customer ID,243007,22.77
Description,4382,0.41
StockCode,0,0.00
Invoice,0,0.00
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Country,0,0.00


In [5]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 34335


In [6]:
df = df.drop_duplicates().copy()

print("Dataset shape after cleaning:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

Dataset shape after cleaning: (1033036, 8)
Remaining duplicates: 0


In [7]:
# Cancellation invoices
cancelled = df[
    df["Invoice"].astype(str).str.startswith("C")
]

# Negative quantity records
returns = df[df["Quantity"] < 0]

# Zero or negative prices
invalid_prices = df[df["Price"] <= 0]

print("Cancelled invoice records:", len(cancelled))
print("Negative quantity records:", len(returns))
print("Non-positive price records:", len(invalid_prices))

Cancelled invoice records: 19104
Negative quantity records: 22496
Non-positive price records: 6019


In [8]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
df = df.rename(columns={
    "invoice": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id"
})

In [9]:
display(df.dtypes.to_frame())

df.head(5)

,0
invoice_no,object
stock_code,object
description,object
quantity,int64
invoice_date,datetime64[us]
price,float64
customer_id,float64
country,str


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
# Date conversion
df["invoice_date"] = pd.to_datetime(
    df["invoice_date"],
    errors="coerce"
)

# Preserve invoice and product codes as strings
df["invoice_no"] = df["invoice_no"].astype("string")
df["stock_code"] = df["stock_code"].astype("string")
df["country"] = df["country"].astype("string")
df["description"] = df["description"].astype("string").str.strip().str.lower()


#strip all string/object column
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].str.strip()

# Customer ID: handle missing values without losing IDs
df["customer_id"] = (
    pd.to_numeric(df["customer_id"], errors="coerce")
    .astype("Int64")
    .astype("string")
)

df.info()

<class 'pandas.DataFrame'>
Index: 1033036 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column        Non-Null Count    Dtype         
---  ------        --------------    -----         
 0   invoice_no    1033036 non-null  string        
 1   stock_code    1033036 non-null  string        
 2   description   1028761 non-null  string        
 3   quantity      1033036 non-null  int64         
 4   invoice_date  1033036 non-null  datetime64[us]
 5   price         1033036 non-null  float64       
 6   customer_id   797885 non-null   string        
 7   country       1033036 non-null  string        
dtypes: datetime64[us](1), float64(1), int64(1), string(5)
memory usage: 70.9 MB


In [11]:
display(df.dtypes.to_frame())

df.head(5)

,0
invoice_no,string
stock_code,string
description,string
quantity,int64
invoice_date,datetime64[us]
price,float64
customer_id,string
country,string


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,white cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [12]:
df["revenue"] = df["quantity"] * df["price"]
df.head(5)

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,489434,79323P,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,489434,79323W,white cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [13]:
df["is_cancelled"] = (
    df["invoice_no"].str.upper().str.startswith("C", na=False)
)


df["is_negative_quantity"] = df["quantity"] < 0
pd.crosstab(
    df["is_cancelled"],
    df["is_negative_quantity"],
    margins=True
)

is_negative_quantity,False,True,All
is_cancelled,,,
False,1010539,3393,1013932
True,1,19103,19104
All,1010540,22496,1033036


In [14]:
df["year"] = df["invoice_date"].dt.year
df["month"] = df["invoice_date"].dt.month

df["year_month"] = (
    df["invoice_date"].dt.to_period("M").astype("string")
)

df["day_name"] = df["invoice_date"].dt.day_name()
df["hour"] = df["invoice_date"].dt.hour
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity,year,month,year_month,day_name,hour
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,False,False,2009,12,2009-12,Tuesday,7
1,489434,79323P,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
2,489434,79323W,white cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,False,False,2009,12,2009-12,Tuesday,7
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,False,False,2009,12,2009-12,Tuesday,7


In [15]:
raw_transactions =df.copy()

valid_base = (
    df["invoice_date"].notna() &
    df["price"].gt(0)
)

sales_mask = (
    valid_base &
    (~df["is_cancelled"]) &
    (df["quantity"] > 0)
)

fact_sales = df.loc[sales_mask].copy()

In [16]:
adjustment_mask = (
    df["is_cancelled"] |
    df["is_negative_quantity"]
)

fact_adjustments = df.loc[adjustment_mask].copy()

fact_adjustments["is_financial_adjustment"] = (
    fact_adjustments["price"].gt(0) &
    fact_adjustments["invoice_date"].notna() &
    fact_adjustments["revenue"].lt(0)
)

In [17]:
gross_sales = fact_sales["revenue"].sum()
negative_adjustments = fact_adjustments.loc[
    fact_adjustments["is_financial_adjustment"],
    "revenue"
].sum()
print("Gross sales :", round(gross_sales,2))
print("negative sales :", round(negative_adjustments,2 ))


Gross sales : 20476260.45
negative sales : -1462424.18


In [18]:
net_sales = gross_sales + negative_adjustments
print("net sales", net_sales)

net sales 19013836.268


In [19]:
print("Fact Adjustments shape : ", fact_adjustments.shape)
print("Fact Adjustments duplicate: ", fact_adjustments.duplicated().sum())
print("Fact Sales shape : ", fact_sales.shape)
print("Fact Sales duplicate: ", fact_sales.duplicated().sum())
print("Raw Transactions shape : ", raw_transactions.shape)
print("Raw Transactions duplicate: ", raw_transactions.duplicated().sum())

Fact Adjustments shape :  (22497, 17)
Fact Adjustments duplicate:  0
Fact Sales shape :  (1007913, 16)
Fact Sales duplicate:  0
Raw Transactions shape :  (1033036, 16)
Raw Transactions duplicate:  0


### Investigating Data Disparity between fact sales and fact adjustments combined with raw transactions total records


In [20]:
print("Raw transactions:", len(df))
print("Fact sales:", len(fact_sales))
print("Fact adjustments:", len(fact_adjustments))

print(
    "Combined:",
    len(fact_sales) + len(fact_adjustments)
)

print(
    "Difference:",
    len(df) - len(fact_sales) - len(fact_adjustments)
)

Raw transactions: 1033036
Fact sales: 1007913
Fact adjustments: 22497
Combined: 1030410
Difference: 2626


In [21]:
classified_index = (
    fact_sales.index.union(fact_adjustments.index)
)

unclassified = df.loc[
    ~df.index.isin(classified_index)
].copy()

print("Unclassified records:", len(unclassified))

Unclassified records: 2626


In [22]:
unclassified.value_counts()

invoice_no  stock_code  description                     quantity  invoice_date         price  customer_id  country         revenue  is_cancelled  is_negative_quantity  year  month  year_month  day_name   hour
489825      22076       6 ribbons empire                12        2009-12-02 13:34:00  0.0    16126        United Kingdom  0.0      False         False                 2009  12     2009-12     Wednesday  13      1
489998      48185       door mat fairy cake             2         2009-12-03 11:19:00  0.0    15658        United Kingdom  0.0      False         False                 2009  12     2009-12     Thursday   11      1
490727      M           manual                          1         2009-12-07 16:38:00  0.0    17231        United Kingdom  0.0      False         False                 2009  12     2009-12     Monday     16      1
490961      22065       christmas pudding trinket pot   1         2009-12-08 15:25:00  0.0    14108        United Kingdom  0.0      False         Fal

In [23]:
print(unclassified[
    [
        "quantity",
        "price",
        "invoice_date",
        "is_cancelled",
        "is_negative_quantity"
    ]
].describe(include="all"))

            quantity         price                invoice_date is_cancelled  \
count    2626.000000   2626.000000                        2626         2626   
unique           NaN           NaN                         NaN            1   
top              NaN           NaN                         NaN        False   
freq             NaN           NaN                         NaN         2626   
mean       95.492765    -60.425034  2010-11-10 00:19:47.296268          NaN   
min         1.000000 -53594.360000         2009-12-01 17:39:00          NaN   
25%         1.000000      0.000000         2010-05-14 11:54:30          NaN   
50%         4.000000      0.000000         2010-10-27 10:00:00          NaN   
75%        24.000000      0.000000         2011-05-05 12:36:00          NaN   
max     12540.000000      0.000000         2011-12-08 14:06:00          NaN   
std       555.984510   1581.021688                         NaN          NaN   

       is_negative_quantity  
count                

In [24]:
zero_price = unclassified[unclassified["price"] == 0].copy()

negative_price = unclassified[unclassified["price"] < 0].copy()

print("Zero-price records:", len(zero_price))
print("Negative-price records:", len(negative_price))
print("Zero-price revenue:", zero_price["revenue"].sum())
print("Negative-price revenue:", negative_price["revenue"].sum())

Zero-price records: 2621
Negative-price records: 5
Zero-price revenue: 0.0
Negative-price revenue: -158676.13999999998


In [25]:
print(
    negative_price[
        [
            "invoice_no",
            "stock_code",
            "description",
            "quantity",
            "price",
            "revenue",
            "invoice_date",
            "customer_id",
            "country"
        ]
    ].head(20).to_string(index=False)
)

invoice_no stock_code     description  quantity     price   revenue        invoice_date customer_id        country
   A506401          B adjust bad debt         1 -53594.36 -53594.36 2010-04-29 13:36:00        <NA> United Kingdom
   A516228          B adjust bad debt         1 -44031.79 -44031.79 2010-07-19 11:24:00        <NA> United Kingdom
   A528059          B adjust bad debt         1 -38925.87 -38925.87 2010-10-20 12:04:00        <NA> United Kingdom
   A563186          B adjust bad debt         1 -11062.06 -11062.06 2011-08-12 14:51:00        <NA> United Kingdom
   A563187          B adjust bad debt         1 -11062.06 -11062.06 2011-08-12 14:52:00        <NA> United Kingdom


In [26]:
print(
    unclassified["invoice_no"]
    .astype(str)
    .str[:1]
    .value_counts(dropna=False)
)

invoice_no
5    2242
4     379
A       5
Name: count, dtype: int64


In [27]:
bad_debt = unclassified[
    unclassified["description"]
    .str.contains("adjust bad debt", case=False, na=False)
].copy()

print("Bad debt records:", len(bad_debt))
print("Total bad debt amount:", bad_debt["revenue"].sum())

print(
    bad_debt[
        ["invoice_no", "description", "quantity",
         "price", "revenue", "invoice_date"]
    ].head(20)
)

Bad debt records: 5
Total bad debt amount: -158676.13999999998
       invoice_no      description  quantity     price   revenue  \
179403    A506401  adjust bad debt         1 -53594.36 -53594.36   
276274    A516228  adjust bad debt         1 -44031.79 -44031.79   
403472    A528059  adjust bad debt         1 -38925.87 -38925.87   
825444    A563186  adjust bad debt         1 -11062.06 -11062.06   
825445    A563187  adjust bad debt         1 -11062.06 -11062.06   

              invoice_date  
179403 2010-04-29 13:36:00  
276274 2010-07-19 11:24:00  
403472 2010-10-20 12:04:00  
825444 2011-08-12 14:51:00  
825445 2011-08-12 14:52:00  


In [28]:
# Identify bad-debt records
bad_debt = unclassified[
    unclassified["description"]
    .str.contains("adjust bad debt", case=False, na=False)
].copy()

# Label existing adjustments
fact_adjustments = fact_adjustments.copy()

fact_adjustments["adjustment_type"] = "Other Adjustment"

fact_adjustments.loc[
    fact_adjustments["is_cancelled"],
    "adjustment_type"
] = "Cancellation"

fact_adjustments.loc[
    fact_adjustments["is_negative_quantity"] &
    (~fact_adjustments["is_cancelled"]),
    "adjustment_type"
] = "Return / Negative Quantity"

# Label bad debt records
bad_debt["adjustment_type"] = "Bad Debt"
bad_debt["is_financial_adjustment"] = (
    bad_debt["revenue"].lt(0) &
    bad_debt["invoice_date"].notna()
)

# Append bad debt to adjustments
fact_adjustments = pd.concat(
    [fact_adjustments, bad_debt],
    ignore_index=True
)
print(fact_adjustments.shape)
fact_adjustments.head(5)


(22502, 18)


,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity,year,month,year_month,day_name,hour,is_financial_adjustment,adjustment_type
0,C489449,22087,paper bunting white lace,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4,True,True,2009,12,2009-12,Tuesday,10,True,Cancellation
1,C489449,85206A,cream felt easter egg basket,-6,2009-12-01 10:33:00,1.65,16321,Australia,-9.9,True,True,2009,12,2009-12,Tuesday,10,True,Cancellation
2,C489449,21895,potting shed sow 'n' grow set,-4,2009-12-01 10:33:00,4.25,16321,Australia,-17.0,True,True,2009,12,2009-12,Tuesday,10,True,Cancellation
3,C489449,21896,potting shed twine,-6,2009-12-01 10:33:00,2.10,16321,Australia,-12.6,True,True,2009,12,2009-12,Tuesday,10,True,Cancellation
4,C489449,22083,paper chain kit retro spot,-12,2009-12-01 10:33:00,2.95,16321,Australia,-35.4,True,True,2009,12,2009-12,Tuesday,10,True,Cancellation


In [29]:
print("Before append:", len(fact_adjustments) - len(bad_debt))
print("Bad debt:", len(bad_debt))
print("After append:", len(fact_adjustments))

Before append: 22497
Bad debt: 5
After append: 22502


In [30]:
zero_price = raw_transactions[raw_transactions["price"] == 0].copy()

print("Zero-price records:", len(zero_price))

Zero-price records: 6014


In [31]:
zero_price_all = df[df["price"] == 0].copy()

zero_price_unclassified = unclassified[
    unclassified["price"] == 0
].copy()

zero_price_sales = fact_sales[
    fact_sales["price"] == 0
].copy()

zero_price_adjustments = fact_adjustments[
    fact_adjustments["price"] == 0
].copy()

print("Zero price in original df:", len(zero_price_all))
print("Zero price in unclassified:", len(zero_price_unclassified))
print("Zero price in fact_sales:", len(zero_price_sales))
print("Zero price in fact_adjustments:", len(zero_price_adjustments))

Zero price in original df: 6014
Zero price in unclassified: 2621
Zero price in fact_sales: 0
Zero price in fact_adjustments: 3393


In [32]:
zero_price["description"].value_counts(dropna=False).head(50)

description
<NA>                                  4275
check                                  163
damaged                                 99
?                                       90
damages                                 88
found                                   38
missing                                 30
sold as set on dotcom                   20
amazon                                  18
adjustment                              18
dotcom                                  15
owl doorstop                            14
polyester filler pad 45x45cm            12
polyester filler pad 40x40cm            10
smashed                                  9
mailout                                  9
french blue metal door sign 1            9
thrown away                              9
unsaleable, destroyed.                   9
picnic basket wicker large               8
checked                                  8
given away                               8
box of 24 cocktail parasols              8

In [34]:
zero_price["is_cancelled"].value_counts(dropna=False)

is_cancelled
False    6014
Name: count, dtype: Int64

In [35]:
zero_price[
    [
        "invoice_no",
        "stock_code",
        "description",
        "quantity",
        "price",
        "revenue",
        "invoice_date",
        "customer_id",
        "country",
        "is_cancelled"
    ]
].head(20)

,invoice_no,stock_code,description,quantity,price,revenue,invoice_date,customer_id,country,is_cancelled
263,489464,21733,85123a mixed,-96,0.0,-0.0,2009-12-01 10:52:00,<NA>,United Kingdom,False
283,489463,71477,short,-240,0.0,-0.0,2009-12-01 10:52:00,<NA>,United Kingdom,False
284,489467,85123A,21733 mixed,-192,0.0,-0.0,2009-12-01 10:53:00,<NA>,United Kingdom,False
470,489521,21646,<NA>,-50,0.0,-0.0,2009-12-01 11:44:00,<NA>,United Kingdom,False
3114,489655,20683,<NA>,-44,0.0,-0.0,2009-12-01 17:26:00,<NA>,United Kingdom,False
3161,489659,21350,<NA>,230,0.0,0.0,2009-12-01 17:39:00,<NA>,United Kingdom,False
3162,489660,35956,lost,-1043,0.0,-0.0,2009-12-01 17:43:00,<NA>,United Kingdom,False
3168,489663,35605A,damages,-117,0.0,-0.0,2009-12-01 18:02:00,<NA>,United Kingdom,False
3731,489781,84292,<NA>,17,0.0,0.0,2009-12-02 11:45:00,<NA>,United Kingdom,False
4296,489806,18010,<NA>,-770,0.0,-0.0,2009-12-02 12:42:00,<NA>,United Kingdom,False


In [38]:
df["is_bad_debt"] = (
    df["description"] == "adjust bad debt"
)
df["is_zero_price"] = df["price"].eq(0)
df.head()

,invoice_no,stock_code,description,quantity,invoice_date,price,customer_id,country,revenue,is_cancelled,is_negative_quantity,year,month,year_month,day_name,hour,is_bad_debt,is_zero_price
0,489434,85048,15cm christmas glass ball 20 lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4,False,False,2009,12,2009-12,Tuesday,7,False,False
1,489434,79323P,pink cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7,False,False
2,489434,79323W,white cherry lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0,False,False,2009,12,2009-12,Tuesday,7,False,False
3,489434,22041,"record frame 7"" single size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8,False,False,2009,12,2009-12,Tuesday,7,False,False
4,489434,21232,strawberry ceramic trinket box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0,False,False,2009,12,2009-12,Tuesday,7,False,False


In [41]:
display(df.dtypes.to_frame())

,0
invoice_no,string
stock_code,string
description,string
quantity,int64
invoice_date,datetime64[us]
price,float64
customer_id,string
country,string
revenue,float64
is_cancelled,boolean


In [43]:
print(df[[
    "is_bad_debt",
    "is_cancelled",
    "is_negative_quantity"
]].dtypes)

print("\nMissing values:")
print(df[[
    "is_bad_debt",
    "is_cancelled",
    "is_negative_quantity"
]].isna().sum())

is_bad_debt             boolean
is_cancelled            boolean
is_negative_quantity       bool
dtype: object

Missing values:
is_bad_debt             4275
is_cancelled               0
is_negative_quantity       0
dtype: int64


In [48]:
df["is_bad_debt"] = (
    df["description"]
    .eq("adjust bad debt")
    .fillna(False)
)

df["is_cancelled"] = (
    df["invoice_no"]
    .astype("string")
    .str.startswith("C", na=False)
)

df["is_negative_quantity"] = (
    df["quantity"].lt(0)
)

df["is_zero_price"] = (
    df["price"].eq(0)
)

In [49]:
conditions = [
    df["is_bad_debt"].to_numpy(dtype=bool),

    df["is_cancelled"].to_numpy(dtype=bool),

    df["is_negative_quantity"].to_numpy(dtype=bool),

    (
        df["quantity"].gt(0)
        & df["price"].gt(0)
        & df["invoice_date"].notna()
    ).to_numpy(dtype=bool)
]

choices = [
    "Bad Debt",
    "Cancellation",
    "Return / Negative Quantity",
    "Valid Sale"
]

df["transaction_status"] = np.select(
    conditions,
    choices,
    default="Requires Review"
)

In [50]:
df["transaction_status"].value_counts()

transaction_status
Valid Sale                    1007912
Cancellation                    19104
Return / Negative Quantity       3393
Requires Review                  2621
Bad Debt                            6
Name: count, dtype: int64

In [51]:
df.groupby("transaction_status").agg(
    records=("invoice_no", "size"),
    total_revenue=("revenue", "sum"),
    zero_price_records=("is_zero_price", "sum")
)

,records,total_revenue,zero_price_records
transaction_status,,,
Bad Debt,6,-1.476141e+05,0
Cancellation,19104,-1.462051e+06,0
Requires Review,2621,0.000000e+00,2621
Return / Negative Quantity,3393,0.000000e+00,3393
Valid Sale,1007912,2.046520e+07,0


Unclassified Zero-Price Transactions

A total of 2,621 records (approximately 0.25% of the processed dataset) remained classified as Requires Review. These records had zero unit prices and did not meet the established criteria for valid sales, cancellations, negative-quantity transactions, or bad debt.

Due to their relatively small proportion and the ambiguity of their descriptions, they were retained in the raw transaction dataset but excluded from confirmed sales and adjustment fact tables. No revenue was attributed to these records.

This limitation is acknowledged to preserve transparency and avoid unsupported transaction classifications.

In [53]:
print(df["transaction_status"].value_counts())

print("Total records:", len(df))
print("Unclassified:", 
      (df["transaction_status"] == "Requires Review").sum())

print("Unclassified revenue:",
      df.loc[
          df["transaction_status"] == "Requires Review",
          "revenue"
      ].sum())

transaction_status
Valid Sale                    1007912
Cancellation                    19104
Return / Negative Quantity       3393
Requires Review                  2621
Bad Debt                            6
Name: count, dtype: int64
Total records: 1033036
Unclassified: 2621
Unclassified revenue: 0.0


In [54]:
# REBUILDING THE FACT SALES AND FACT ADJUSTMENTS

fact_sales = df[
    df["transaction_status"] == "Valid Sale"
].copy()

fact_adjustments = df[
    df["transaction_status"].isin([
        "Bad Debt",
        "Cancellation",
        "Return / Negative Quantity"
    ])
].copy()

In [55]:
print("Raw transactions:", len(df))
print("Fact sales:", len(fact_sales))
print("Fact adjustments:", len(fact_adjustments))
print("Requires review:", 
      (df["transaction_status"] == "Requires Review").sum())

print(
    "Reconciled:",
    len(fact_sales) + len(fact_adjustments)
    + (df["transaction_status"] == "Requires Review").sum()
)

Raw transactions: 1033036
Fact sales: 1007912
Fact adjustments: 22503
Requires review: 2621
Reconciled: 1033036


In [56]:
print("Raw revenue:", df["revenue"].sum())
print("Sales revenue:", fact_sales["revenue"].sum())
print("Adjustments revenue:", fact_adjustments["revenue"].sum())

print(
    "Unclassified revenue:",
    df.loc[
        df["transaction_status"] == "Requires Review",
        "revenue"
    ].sum()
)

print(
    "Reconciled revenue:",
    fact_sales["revenue"].sum()
    + fact_adjustments["revenue"].sum()
    + df.loc[
        df["transaction_status"] == "Requires Review",
        "revenue"
    ].sum()
)

Raw revenue: 18855533.698
Sales revenue: 20465198.387999997
Adjustments revenue: -1609664.69
Unclassified revenue: 0.0
Reconciled revenue: 18855533.697999995


In [61]:
print(
    fact_adjustments.groupby("transaction_status").agg(
        records=("invoice_no", "size"),
        total_quantity=("quantity", "sum"),
        total_revenue=("revenue", "sum")
    )
)

                            records  total_quantity  total_revenue
transaction_status                                                
Bad Debt                          6               6     -147614.08
Cancellation                  19104         -476819    -1462050.61
Return / Negative Quantity     3393         -569314           0.00


In [62]:
print(
    fact_adjustments[
        fact_adjustments["transaction_status"] == "Bad Debt"
    ][["invoice_no", "quantity", "price", "revenue"]]
)

       invoice_no  quantity     price   revenue
179403    A506401         1 -53594.36 -53594.36
276274    A516228         1 -44031.79 -44031.79
403472    A528059         1 -38925.87 -38925.87
825443    A563185         1  11062.06  11062.06
825444    A563186         1 -11062.06 -11062.06
825445    A563187         1 -11062.06 -11062.06


In [63]:
df[
    df["invoice_no"].isin([
        "A563185",
        "A563186",
        "A563187"
    ])
][[
    "invoice_no",
    "stock_code",
    "description",
    "quantity",
    "price",
    "revenue",
    "customer_id"
]]

,invoice_no,stock_code,description,quantity,price,revenue,customer_id
825443,A563185,B,adjust bad debt,1,11062.06,11062.06,<NA>
825444,A563186,B,adjust bad debt,1,-11062.06,-11062.06,<NA>
825445,A563187,B,adjust bad debt,1,-11062.06,-11062.06,<NA>


Bad-debt adjustments were identified using the transaction description. All six records were retained, including one positive-value entry. The positive entry was not reclassified because its accounting meaning could not be independently established.

In [64]:
# 1. Check transaction status
print(df["transaction_status"].value_counts(dropna=False))

# 2. Check fact table revenue reconciliation
print(
    round(
        df["revenue"].sum()
        - fact_sales["revenue"].sum()
        - fact_adjustments["revenue"].sum(),
        6
    )
)

# 3. Check sales validity
print(
    (fact_sales["quantity"] <= 0).sum(),
    (fact_sales["price"] <= 0).sum(),
    fact_sales["invoice_date"].isna().sum()
)

# 4. Check duplicate rows
print("Exact duplicate rows:", df.duplicated().sum())

transaction_status
Valid Sale                    1007912
Cancellation                    19104
Return / Negative Quantity       3393
Requires Review                  2621
Bad Debt                            6
Name: count, dtype: int64
0.0
0 0 0
Exact duplicate rows: 0


In [65]:
print(df.dtypes)

invoice_no                      string
stock_code                      string
description                     string
quantity                         int64
invoice_date            datetime64[us]
price                          float64
customer_id                     string
country                         string
revenue                        float64
is_cancelled                   boolean
is_negative_quantity              bool
year                             int32
month                            int32
year_month                      string
day_name                           str
hour                             int32
is_bad_debt                    boolean
is_zero_price                     bool
transaction_status                 str
dtype: object


In [66]:
print("Minimum invoice date:", df["invoice_date"].min())
print("Maximum invoice date:", df["invoice_date"].max())

print("Minimum revenue:", df["revenue"].min())
print("Maximum revenue:", df["revenue"].max())

Minimum invoice date: 2009-12-01 07:45:00
Maximum invoice date: 2011-12-09 12:50:00
Minimum revenue: -168469.6
Maximum revenue: 168469.6


In [67]:
string_columns = [
    "invoice_no",
    "stock_code",
    "description",
    "customer_id",
    "country",
    "year_month",
    "day_name",
    "transaction_status"
]

for col in string_columns:
    print(
        f"{col}:",
        df[col].astype("string").str.len().max()
    )

invoice_no: 7
stock_code: 12
description: 35
customer_id: 5
country: 20
year_month: 7
day_name: 9
transaction_status: 26


In [68]:
print(
    "Price decimal places:",
    df["price"].dropna().map(
        lambda x: len(str(x).split(".")[-1])
    ).value_counts().sort_index()
)

print(
    "Revenue decimal places:",
    df["revenue"].dropna().map(
        lambda x: len(str(x).split(".")[-1])
    ).value_counts().sort_index()
)

Price decimal places: price
1     69791
2    963227
3        18
Name: count, dtype: int64
Revenue decimal places: revenue
1     417241
2     449174
3         18
12         5
13       379
14     10999
15    141225
16     13993
17         2
Name: count, dtype: int64


In [69]:
for col in ["price", "revenue"]:
    difference = (df[col] - df[col].round(4)).abs()

    print(f"\n{col}")
    print("Values changed by rounding to 4 decimals:",
          (difference > 1e-9).sum())
    print("Maximum rounding difference:",
          difference.max())


price
Values changed by rounding to 4 decimals: 0
Maximum rounding difference: 0.0

revenue
Values changed by rounding to 4 decimals: 0
Maximum rounding difference: 9.094947017729282e-13


### Making a Sampled dataset and Import Dataset into SQL Server Database


In [70]:
# Preserve the complete processed dataset
raw_transactions = df.copy()

# Create a stratified sample
sample_raw = (
    pd.concat([
        group.sample(
            n=min(1000, len(group)),
            random_state=42
        )
        for _, group in raw_transactions.groupby(
            "transaction_status",
            sort=False
        )
    ])
    .sort_index()
    .copy()
)

# Add a source row identifier for traceability
sample_raw.insert(0, "source_row_id", sample_raw.index)

# Create fact table samples from the same raw sample
sample_sales = sample_raw[
    sample_raw["transaction_status"] == "Valid Sale"
].copy()

sample_adjustments = sample_raw[
    sample_raw["transaction_status"].isin([
        "Bad Debt",
        "Cancellation",
        "Return / Negative Quantity"
    ])
].copy()

# Organize the three datasets
sample_tables = {
    "raw_transactions_sample": sample_raw,
    "fact_sales_sample": sample_sales,
    "fact_adjustments_sample": sample_adjustments
}


In [72]:

for table_name, table in sample_tables.items():
    print(f"{table_name}: {len(table):,} rows")

print("\nTransaction status distribution:")
print(sample_raw["transaction_status"].value_counts())

# Check source row ID uniqueness
assert sample_raw["source_row_id"].is_unique

# Check fact tables are subsets of raw sample
assert set(sample_sales["source_row_id"]).issubset(
    set(sample_raw["source_row_id"])
)

assert set(sample_adjustments["source_row_id"]).issubset(
    set(sample_raw["source_row_id"])
)

# Check row reconciliation
assert (
    len(sample_sales)
    + len(sample_adjustments)
    + (sample_raw["transaction_status"] == "Requires Review").sum()
    == len(sample_raw)
)

print("\nAll sample validation checks passed.")


raw_transactions_sample: 4,006 rows
fact_sales_sample: 1,000 rows
fact_adjustments_sample: 2,006 rows

Transaction status distribution:
transaction_status
Cancellation                  1000
Return / Negative Quantity    1000
Valid Sale                    1000
Requires Review               1000
Bad Debt                         6
Name: count, dtype: int64

All sample validation checks passed.


In [71]:

from pathlib import Path

output_dir = Path("../Data/Processed")
output_dir.mkdir(parents=True, exist_ok=True)

for table_name, table in sample_tables.items():
    file_path = output_dir / f"{table_name}.csv"

    table.to_csv(
        file_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Exported: {file_path}")


Exported: ..\Data\Processed\raw_transactions_sample.csv
Exported: ..\Data\Processed\fact_sales_sample.csv
Exported: ..\Data\Processed\fact_adjustments_sample.csv


In [73]:
full_tables = {
    "raw_transactions": df,
    "fact_sales": fact_sales,
    "fact_adjustments": fact_adjustments
}

In [74]:
from sqlalchemy import (
    BigInteger,
    String,
    Unicode,
    Integer,
    Numeric,
    Boolean
)
from sqlalchemy.dialects import mssql

sql_dtypes = {
    "invoice_no": String(20),
    "stock_code": String(50),
    "description": Unicode(255),
    "quantity": Integer(),
    "invoice_date": mssql.DATETIME2(precision=6),
    "price": Numeric(18, 4),
    "customer_id": String(20),
    "country": Unicode(100),
    "revenue": Numeric(18, 4),
    "is_cancelled": Boolean(),
    "is_negative_quantity": Boolean(),
    "year": Integer(),
    "month": Integer(),
    "year_month": String(7),
    "day_name": String(15),
    "hour": Integer(),
    "is_bad_debt": Boolean(),
    "is_zero_price": Boolean(),
    "transaction_status": String(50)
}

In [75]:
from sqlalchemy import create_engine, URL

# SQL Server connection
connection_url = URL.create(
    "mssql+pyodbc",
    host="localhost",
    database="online_retail",
    query={
        "driver": "ODBC Driver 18 for SQL Server",
        "trusted_connection": "yes",
        "TrustServerCertificate": "yes"
    }
)

engine = create_engine(
    connection_url,
    fast_executemany=True
)

# Import directly into SQL Server
for table_name, dataframe in full_tables.items():

    sql_dataframe = dataframe.copy()

    # Handle pandas nullable string columns
    for col in sql_dataframe.select_dtypes(
        include=["string"]
    ).columns:
        sql_dataframe[col] = (
            sql_dataframe[col]
            .astype(object)
            .where(sql_dataframe[col].notna(), None)
        )

    sql_dataframe.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000,
        dtype=sql_dtypes
    )

    print(f"Imported {table_name}: {len(dataframe):,} rows")

print("All tables imported successfully!")

Imported raw_transactions: 1,033,036 rows
Imported fact_sales: 1,007,912 rows
Imported fact_adjustments: 22,503 rows
All tables imported successfully!


## SQL Server Export / Import Utilities

This section contains optional database export/import scripts and archival code used for loading processed datasets into SQL Server. It is intentionally separated from the main data-cleaning and analysis workflow and may be commented out or reused as needed.

In [ ]:
for table_name, dataframe in full_tables.items():

    sql_dataframe = dataframe.copy()

    # Handle pandas nullable string columns
    for col in sql_dataframe.select_dtypes(
        include=["string"]
    ).columns:

        sql_dataframe[col] = (
            sql_dataframe[col]
            .astype(object)
            .where(sql_dataframe[col].notna(), None)
        )

    sql_dataframe.to_sql(
        name=table_name,
        con=engine,
        if_exists="replace",
        index=False,
        chunksize=5000,
        dtype=sql_dtypes
    )

    print(
        f"Imported {table_name}: "
        f"{len(sql_dataframe):,} rows"
    )

In [ ]:
# Making a sample processed dataset for GitHub Repositories
import os

output_dir = os.path.join("..", "Data", "Processed")
os.makedirs(output_dir, exist_ok=True)

for table_name, dataframe in {
    "raw_transactions": raw_transactions,
    "fact_sales": fact_sales,
    "fact_adjustments": fact_adjustments,
}.items():
    sample_path = os.path.join(output_dir, f"{table_name}_sample.csv")
    
    # Export 1,000 rows without the pandas row index
    dataframe.head(1000).to_csv(sample_path, index=False)
    
    size_kb = os.path.getsize(sample_path) / 1024
    print(f"Saved {sample_path} ({size_kb:.1f} KB)")

In [ ]:
# raw_transactions =df.copy()
# raw_transactions.to_csv("..\\Data\\Processed\\raw_transactions.csv", index=False)
# fact_sales.to_csv("..\\Data\\Processed\\fact_sales.csv", index=False)
# fact_adjustments.to_csv("..\\Data\\Processed\\fact_adjustments.csv", index=False)

In [ ]:
# from sqlalchemy import create_engine, text
# from urllib.parse import quote_plus



# server = "localhost"  # e.g. localhost or SERVER\\INSTANCE
# database = "online_retail"
# driver = "ODBC Driver 18 for SQL Server"  # Must be installed on this machine

# odbc_connection = (
#     f"DRIVER={{{driver}}};"
#     f"SERVER={server};"
#     f"DATABASE={database};"
#     "Trusted_Connection=yes;"
#     "TrustServerCertificate=yes;"
# )

# connection_url = (
#     "mssql+pyodbc:///?odbc_connect="
#     + quote_plus(odbc_connection)
# )

# engine = create_engine(connection_url, fast_executemany=True)

# # Check the connection
# with engine.connect() as connection:
#     print(connection.execute(text("SELECT 1")).scalar())

# # Load the three dataframes into dbo tables
# for table_name, dataframe in {
#     "raw_transactions": raw_transactions,
#     "fact_sales": fact_sales,
#     "fact_adjustments": fact_adjustments,
# }.items():
#     dataframe.to_sql(
#         name=table_name,
#         con=engine,
#         schema="dbo",
#         if_exists="replace",  # Change to "append" to keep existing table data
#         index=False,
#         chunksize=5000,
#         method=None,
#     )
#     print(f"Loaded {len(dataframe):,} rows into dbo.{table_name}")